In [ ]:
#Install requirements
%pip install -r "../requirements.txt"

In [ ]:
#Import required libraries
import cv2
import numpy as np
from scipy import stats
from skimage import filters
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import os
import gc
import sys

In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
# BATCH PROCESSING FUNCTION

def batch_process_hierarchical(quantifier, root_dir, markers, regions,  groups,
                               threshold_method, threshold_param, tissue_threshold): 
    """
    Process images in hierarchical folder structure and save results.
    
    Folder structure:
    root_dir/
        MARKER/
            REGION/
                GROUP/
                    SUBJECT/
                        DAB_RGB_Verification/
                            image1.jpg
                            image2.jpg
                            ...
                            Calculated_Masks_Final/  
                                mask_image1.jpg
                                mask_image2.jpg
                                ...
    
    Output CSVs:
    root_dir/MARKER/REGION_GROUP_Analysis_Results.csv
    """
    
    for marker in markers:
        marker_path = os.path.join(root_dir, marker)
        if not os.path.exists(marker_path):
            print(f"\n Skipping marker: {marker} (Path not found)")
            continue
        
        print(f"\n{'='*80}")
        print(f"PROCESSING MARKER: {marker}")
        print(f"{'='*80}")
        
        for region in regions:
            region_path = os.path.join(marker_path, region)
            if not os.path.exists(region_path):
                print(f"\nSkipping region: {region} (Region not found)")
                continue
            print(f"\n{'='*80}")
            print(f"PROCESSING REGION: {region}")
            print(f"{'='*80}")

            for group in groups:
                group_path = os.path.join(region_path, 'Deconvolved_Results',  group)
                if not os.path.exists(group_path):
                    print(f"  Skipping {group} (Path not found)")
                    continue
                
                print(f"\n--- Processing:  {group} ---")
                
                all_data_for_group = []
                
                # Find subject folders
                subjects = sorted([
                    s for s in os.listdir(group_path) 
                    if os.path.isdir(os.path.join(group_path, s)) 
                    and s.startswith(('A', 'PDC'))
                ])
                
                if not subjects:
                    print(f"No subjects found in {group_path}")
                    continue
                
                print(f"  Found {len(subjects)} subjects: {', '.join(subjects)}")
                
                for sub in subjects:
                    print(f"\n  Processing subject: {sub}")
                    
                    # Input/output folders
                    input_folder = os.path.join(group_path, sub, 'DAB_RGB_Verification')
                    output_mask_folder = os.path.join(input_folder, 'Calculated_Masks_Final')
                    
                    if not os.path.exists(input_folder):
                        print(f"Missing DAB_RGB_Verification folder")
                        continue
                    
                    os.makedirs(output_mask_folder, exist_ok=True)
                    
                    # Find image files
                    files = [
                        f for f in os.listdir(input_folder) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))
                        and not f.startswith('mask_')  # Skip existing masks
                    ]
                    
                    if not files:
                        print(f"No images found")
                        continue
                    
                    print(f" Processing {len(files)} images...")
                    
                    processed_count = 0
                    for filename in files:
                        img_path = os.path.join(input_folder, filename)
                        
                        # Quantify image
                        result = quantifier.quantify_image(
                            img_path,
                            threshold_method=threshold_method,
                            threshold_param=threshold_param,
                            tissue_mask_threshold=tissue_threshold
                        )
                        
                        if result:
                            # Save mask
                            mask_path = os.path.join(output_mask_folder, f"mask_{filename}")
                            save_mask_visualization(img_path, result, mask_path, 
                                                    threshold_param, tissue_threshold, quantifier)
                            
                            # Store data for CSV
                            all_data_for_group.append({
                                'Image_Title': filename,
                                'Subject_ID': sub,
                                'Area_Percent': result['area_percent'],
                                'Mean_Positive_Intensity': result['mean_positive_intensity'],
                                'DAB_Density': result['dab_density'],
                                'Positive_Pixels': result['positive_pixels']
                                })
                            
                            processed_count += 1
                    
                    print(f"Processed: {processed_count}/{len(files)} images")
                    
                    # Clear memory
                    gc.collect()
                
                # Save CSV for this marker/region/group combination
                if all_data_for_group:
                    save_wide_format_csv(all_data_for_group, marker_path, region_path,  group)
                else:
                    print(f"No data collected for {group}")
    
    print(f"\n{'='*80}")
    print("✓ BATCH PROCESSING COMPLETE")
    print(f"{'='*80}\n")


def save_mask_visualization(img_path, result, output_path, threshold_param, 
                           tissue_threshold, quantifier):
    """
    Create and save mask visualization.
    White = tissue, Red = DAB positive, Black = background
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Recreate masks
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    tissue_mask = gray < tissue_threshold
    
    img_od = -np.log((img_rgb.astype(np.float32) / 255.0) + 1e-6)
    dab_od = np.dot(img_od, quantifier.dab_vector)
    dab_signal = np.clip(dab_od - quantifier.reference_background, 0, None)
    
    threshold = result['threshold_value']
    dab_positive = (dab_signal > threshold) & tissue_mask
    
    # Create visualization: White=tissue, Red=positive, Black=background
    mask_vis = np.zeros((*dab_positive.shape, 3), dtype=np.uint8)
    mask_vis[tissue_mask] = [255, 255, 255]  # White = tissue
    mask_vis[dab_positive] = [255, 0, 0]  # Red = DAB positive
    
    cv2.imwrite(output_path, cv2.cvtColor(mask_vis, cv2.COLOR_RGB2BGR))


def save_wide_format_csv(data_list, marker_path,  group): #region was removed
    """
    Save results in wide format with each subject having its own tile column.
    Each subject gets 4 columns: Tile | Area_Percent | Mean_Intensity | DAB_Density
    """
    df_long = pd.DataFrame(data_list)
    
    # Get unique subjects
    subjects = sorted(df_long['Subject_ID'].unique())
    
    # Build dataframe for each subject
    subject_dfs = []
    max_tiles = 0
    
    for subject in subjects:
        # Filter data for this subject
        sub_data = df_long[df_long['Subject_ID'] == subject].copy()
        sub_data = sub_data.sort_values('Image_Title').reset_index(drop=True)
        
        # Create columns for this subject
        sub_df = pd.DataFrame({
            f'{subject}_Tile': sub_data['Image_Title'].values,
            f'{subject}_Area_Percent': sub_data['Area_Percent'].values,
            f'{subject}_Mean_Intensity': sub_data['Mean_Positive_Intensity'].values,
            f'{subject}_DAB_Density': sub_data['DAB_Density'].values
        })
        
        subject_dfs.append(sub_df)
        max_tiles = max(max_tiles, len(sub_df))
    
    # Pad dataframes to same length 
    for i in range(len(subject_dfs)):
        if len(subject_dfs[i]) < max_tiles:
            padding = max_tiles - len(subject_dfs[i])
            pad_df = pd.DataFrame(
                [[None] * len(subject_dfs[i].columns)] * padding,
                columns=subject_dfs[i].columns
            )
            subject_dfs[i] = pd.concat([subject_dfs[i], pad_df], ignore_index=True)
    
    # Concatenate all subjects horizontally
    df_wide = pd.concat(subject_dfs, axis=1)
    
    # Save to CSV
    output_csv = os.path.join(marker_path, f"{group}_Analysis_Results.csv")
    df_wide.to_csv(output_csv, index=False)
    
    print(f"\n CSV saved: {output_csv}")
    print(f"Subjects: {len(subjects)}")
    print(f"Max tiles per subject: {max_tiles}")
    print(f"Columns: {len(subjects)} subjects × 4 = {len(df_wide.columns)} columns")

In [ ]:
# CONFIGURATION

ROOT_DIR = r'main_root' #Adjust the path root as needed
MARKERS = ['AQP4', 'GFAP'] #Adjust as needed
REGIONS = ['GM', 'WM']
GROUPS = ['Controls','Patients']

# Threshold settings
THRESHOLD_METHOD = 'fixed_adaptive'
THRESHOLD_PARAM = 3.5  
TISSUE_MASK_THRESHOLD = 245

# Calibration file
CALIBRATION_FILE = r'../data/background_calibration.npz'

#Import function
from src.functions import DABQuantifier

In [ ]:
# MAIN EXECUTION

if __name__ == "__main__":
    
    print("\n" + "="*80)
    print("DAB QUANTIFICATION - BATCH PROCESSING")
    print("="*80)
    
    # 1. Initialize quantifier and load calibration
    print("\n1. Loading calibration...")
    quantifier = DABQuantifier()
    
    try:
        quantifier.load_calibration(CALIBRATION_FILE)
    except FileNotFoundError:
        print(f"\n✗ ERROR: Calibration file not found: {CALIBRATION_FILE}")
        print("Please run calibration first:")
        exit(1)
    
    # 2. Display settings
    print("\n2. Processing settings:")
    print(f"   Root directory: {ROOT_DIR}")
    print(f"   Markers: {', '.join(MARKERS)}")
    print(f"   Regions: {', '.join(REGIONS)}")
    print(f"   Groups: {', '.join(GROUPS)}")
    print(f"   Threshold method: {THRESHOLD_METHOD}")
    print(f"   Threshold parameter: {THRESHOLD_PARAM}")
    print(f"   Tissue mask threshold: {TISSUE_MASK_THRESHOLD}")
    
    # 3. Confirm before proceeding
    response = input("\nProceed with batch processing? (yes/no): ")
    if response.lower() not in ['yes', 'y']:
        print("Aborted.")
        exit(0)
    
    # 4. Run batch processing
    batch_process_hierarchical(
        quantifier=quantifier,
        root_dir=ROOT_DIR,
        markers=MARKERS,
        regions=REGIONS,
        groups=GROUPS,
        threshold_method=THRESHOLD_METHOD,
        threshold_param=THRESHOLD_PARAM,
        tissue_threshold=TISSUE_MASK_THRESHOLD
    )
    
    print("\n" + "="*80)
    print("✓ ALL PROCESSING COMPLETE!")
    print("="*80)